# CleitonForge: Phase Convention Bugs in Quantum Simulators

This notebook demonstrates a key finding from the **Quantum Gate Convention Standard (QGCS)**:

> Quantum Volume (QV) heavy-output scores are **exactly blind** to Rz sign convention bugs.  
> An Rz sign inversion produces `U_wrong = U_ref*`, leaving all measurement probabilities algebraically identical.

We reproduce this finding using `cleitonforge`, a neutral Rust-backed benchmarking layer.

**Install:** `pip install cleitonforge`  
**Paper:** https://doi.org/10.5281/zenodo.21210972  
**GitHub:** https://github.com/cleitonaugusto/CleitonForge

In [ ]:
import cleitonforge as cf
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print(f"cleitonforge loaded. DEFAULT_SEED = {cf.DEFAULT_SEED}")

## 1. The Rz Convention

**OpenQASM 3 (correct):**
$$R_z(\lambda) = \begin{pmatrix} e^{-i\lambda/2} & 0 \\ 0 & e^{+i\lambda/2} \end{pmatrix}$$

**Inverted convention (wrong):**
$$R_z^{\text{wrong}}(\lambda) = \begin{pmatrix} e^{+i\lambda/2} & 0 \\ 0 & e^{-i\lambda/2} \end{pmatrix} = R_z(\lambda)^*$$

### Phase-sensitive witness circuit

Apply $H$ then $R_z(\pi/2)$ to $|0\rangle$. The amplitude ratio `sv[1]/sv[0]` must equal $+i$ (correct) or $-i$ (inverted).

In [ ]:
def amplitude_ratio(backend):
    """sv[1]/sv[0] for H; Rz(π/2)|0⟩ — the QGCS Rz sign witness."""
    c = cf.Circuit(1)
    c.h(0)
    c.rz(math.pi / 2, 0)
    r = cf.run(c, backend=backend)
    sv = [complex(re, im) for re, im in r.statevector]
    return sv[1] / sv[0]

ratio_native  = amplitude_ratio("statevector")
ratio_quantrs2 = amplitude_ratio("quantrs2")

print(f"statevector  sv[1]/sv[0] = {ratio_native:.6f}   ← expected +i (OpenQASM 3 ✅)")
print(f"quantrs2     sv[1]/sv[0] = {ratio_quantrs2:.6f}   ← got -i (sign inverted ❌)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, (name, ratio, color, label) in zip(axes, [
    ("statevector (correct)",  ratio_native,   "#2ecc71", "+i  ✅"),
    ("quantrs2 (inverted)",    ratio_quantrs2, "#e74c3c", "−i  ❌"),
]):
    theta = np.linspace(0, 2 * np.pi, 300)
    ax.plot(np.cos(theta), np.sin(theta), "k-", lw=0.5, alpha=0.3)
    ax.axhline(0, color="k", lw=0.4, alpha=0.3)
    ax.axvline(0, color="k", lw=0.4, alpha=0.3)
    ax.annotate("", xy=(ratio.real, ratio.imag), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect("equal")
    ax.set_title(f"{name}\nsv[1]/sv[0] = {label}", fontsize=10)
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")

fig.suptitle("Rz(π/2)|+⟩ amplitude ratio — phase convention comparison", fontsize=12)
plt.tight_layout()
plt.savefig("rz_phase_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved rz_phase_comparison.png")

## 2. Why Quantum Volume Cannot Detect This Bug

**Theorem:** For circuits built from $R_z$, $R_y$, and CX:
$$U_{\text{wrong}} = U_{\text{ref}}^* \implies |\langle x|U_{\text{wrong}}|0\rangle|^2 = |\langle x|U_{\text{ref}}|0\rangle|^2$$

All QV heavy-output probabilities are **algebraically identical**. We verify this below.

In [ ]:
import random

def random_rz_ry_cx_circuit(n_qubits, n_layers, seed=42):
    """Random circuit using only Rz, Ry, CX — the QV gate set."""
    rng = random.Random(seed)
    c = cf.Circuit(n_qubits)
    for _ in range(n_layers):
        for q in range(n_qubits):
            c.rz(rng.uniform(0, 2 * math.pi), q)
            c.ry(rng.uniform(0, math.pi), q)
        for q in range(0, n_qubits - 1, 2):
            c.cx(q, q + 1)
    return c

def hog_fraction(circuit, backend):
    """Heavy-output generation fraction for a circuit on a given backend."""
    # Ideal probabilities from native
    ideal = cf.run(circuit, backend="statevector")
    probs = ideal.probabilities
    median = sorted(probs)[len(probs) // 2]
    heavy_set = [p > median for p in probs]
    # Backend probabilities
    result = cf.run(circuit, backend=backend)
    return sum(p for p, h in zip(result.probabilities, heavy_set) if h)

print("Rz+Ry+CX circuits — HOG fractions (higher = better, threshold 2/3)")
print(f"{'n':>4}  {'native':>10}  {'quantrs2':>10}  {'match?':>8}")
print("-" * 42)

for n in [2, 3, 4]:
    c = random_rz_ry_cx_circuit(n, n_layers=n, seed=2024)
    hog_nat = hog_fraction(c, "statevector")
    hog_q2  = hog_fraction(c, "quantrs2")
    match = "✅ identical" if abs(hog_nat - hog_q2) < 1e-10 else f"Δ={abs(hog_nat-hog_q2):.2e}"
    print(f"{n:>4}  {hog_nat:>10.6f}  {hog_q2:>10.6f}  {match:>8}")

print("\nConclusion: QV HOG is blind to the sign inversion. 0.0000 difference.")

## 3. QGCS: Running the Conformance Suite

The **Quantum Gate Convention Standard** (QGCS) uses amplitude-level checks that *can* detect the sign bug.

In [ ]:
for backend_name in ["statevector", "quantrs2"]:
    results = cf.certify(backend_name)
    passed = sum(1 for r in results if r.passed)
    total = len(results)
    compliance = "✅ OpenQASM 3 compliant" if passed == total else "❌ NOT OpenQASM 3 compliant"
    
    print(f"\n{'═'*60}")
    print(f" Backend: {backend_name}")
    print(f"{'═'*60}")
    for r in results:
        icon = "✅" if r.passed else "❌"
        detail = f"  → {r.detail}" if r.detail else ""
        print(f"  {icon} [{r.dimension:11}] {r.name}{detail}")
    print(f"{'─'*60}")
    print(f"  Result: {passed}/{total} — {compliance}")

In [ ]:
# Visual comparison of QGCS results
backends = ["statevector", "quantrs2"]
all_results = {b: cf.certify(b) for b in backends}
checks = [r.name for r in all_results["statevector"]]
dimensions = [r.dimension for r in all_results["statevector"]]

fig, ax = plt.subplots(figsize=(11, 5))
colors = {"statevector": "#2ecc71", "quantrs2": "#e74c3c"}
offsets = {"statevector": -0.18, "quantrs2": 0.18}

for backend in backends:
    vals = [1.0 if r.passed else 0.0 for r in all_results[backend]]
    x = np.arange(len(checks))
    bars = ax.bar(x + offsets[backend], vals, width=0.34,
                  color=colors[backend], alpha=0.85,
                  label=f"{backend} ({sum(v>0 for v in vals)}/{len(vals)})")

ax.set_xticks(np.arange(len(checks)))
ax.set_xticklabels(
    [f"{d}\n{n[:22]}" for d, n in zip(dimensions, checks)],
    rotation=45, ha="right", fontsize=7
)
ax.axhline(0.5, color="k", lw=0.5, ls="--", alpha=0.3)
ax.set_yticks([0, 1])
ax.set_yticklabels(["FAIL", "PASS"])
ax.set_ylim(-0.1, 1.3)
ax.set_title("QGCS v0.1 — 12 Phase-Sensitive Conformance Checks", fontsize=12, fontweight="bold")
ax.legend(loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("qgcs_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved qgcs_results.png")

## 4. The P vs Rz Failure: Fidelity = 0

The second QGCS failure for `quantrs2` is the most dramatic: the inverted Rz makes
`P(π/2)|+⟩` and `Rz(π/2)|+⟩` **orthogonal** (fidelity = 0, not a near-miss).

This happens because $P(\pi/2) = e^{i\pi/4} R_z(\pi/2)$ (global phase), but with the
inverted convention, `quantrs2` computes $R_z^{\text{wrong}}(\pi/2)$ and the relationship breaks:
$\langle P|R_z^{\text{wrong}}\rangle = 0$.

In [ ]:
def fidelity(sv_a, sv_b):
    inner = sum(ca.conjugate() * cb for ca, cb in zip(sv_a, sv_b))
    return abs(inner) ** 2

def statevector(circuit, backend):
    r = cf.run(circuit, backend=backend)
    return [complex(re, im) for re, im in r.statevector]

def p_vs_rz_fidelity(backend):
    c_p = cf.Circuit(1); c_p.h(0); c_p.p(math.pi / 2, 0)
    c_rz = cf.Circuit(1); c_rz.h(0); c_rz.rz(math.pi / 2, 0)
    return fidelity(statevector(c_p, backend), statevector(c_rz, backend))

for b in ["statevector", "quantrs2"]:
    f = p_vs_rz_fidelity(b)
    flag = "✅" if abs(f - 1.0) < 1e-6 else "❌"
    print(f"{flag} {b:15}  fidelity(P(π/2)|+⟩, Rz(π/2)|+⟩) = {f:.6f}")

print("\nFor quantrs2: fidelity = 0.0 — the states are orthogonal.")
print("This is the maximum possible error, not a rounding issue.")

## 5. Summary

| Test type | Detects Rz sign bug? |
|-----------|---------------------|
| Quantum Volume (HOG) | ❌ Exactly blind — algebraic identity |
| Probability comparison | ❌ Blind for same reason |
| QGCS amplitude ratio | ✅ Detects immediately |
| QGCS fidelity (P vs Rz) | ✅ Fidelity = 0 for quantrs2 |

**Run `cforge certify --backend <name>` to check any Rust backend.**  
**Call `cf.certify("<name>")` from Python.**

---

*CleitonForge — neutral benchmarking for quantum simulators*  
*MIT License | https://github.com/cleitonaugusto/CleitonForge*